# Loan Risk Scorer: live demo app
This short notebook opens the **Loan Risk Scorer** web app for the presentation.
It does **not** retrain anything. It loads the model saved by `Loan_Default_Prediction.ipynb`.

**Before you start:** run `Loan_Default_Prediction.ipynb` once (Run all) in Colab. That creates `loan_risk_app.joblib` in the same folder, made with Colab's own library versions. Do this again if Colab ever shows an error loading the file.

**Then:** Runtime → Run all here. Click the link ending in **gradio.live** to open the app in a full browser tab.

### Step 1: Go to the project folder (Colab only; it is skipped on your own computer)

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Capstone")   # change this if your folder has a different name
except ImportError:
    pass  # running on your own computer: the notebook is already in the project folder
if not os.path.exists("loan_risk_app.joblib") and os.path.exists("data/loan_risk_app.joblib"):
    os.chdir("data")   # repository layout
print("Working folder:", os.getcwd())
print("App file found:", os.path.exists("loan_risk_app.joblib"))

### Step 2: Install the web-app library

In [ ]:
%pip install -q gradio

### Step 3: Load the saved model

In [ ]:
import joblib, pandas as pd
try:
    app_file = joblib.load("loan_risk_app.joblib")
except Exception as e:
    raise SystemExit("Could not open loan_risk_app.joblib. It was made with different library versions.\n"
                     "Fix: open Loan_Default_Prediction.ipynb in this same folder, click Runtime -> Run all "
                     "(it re-creates the file), then run this notebook again.\nDetails: " + repr(e))
model, DEFAULTS = app_file["model"], app_file["defaults"]
NUM, CAT = app_file["numeric_features"], app_file["categorical_features"]
LOW_CUT, HIGH_CUT, DISTRICTS = app_file["low_cut"], app_file["high_cut"], app_file["district_stats"]
print(f"Model loaded. Test F1 score = {app_file['test_f1']} (pass mark 0.50)")

def score(district, amount, duration, balance_mean, balance_min, balance_last, income,
          overdrawn, penalties, age, card):
    r = dict(DEFAULTS)
    d = DISTRICTS.loc[district]
    r.update(amount=amount, duration=int(duration), balance_mean=balance_mean, balance_min=balance_min,
             balance_last=balance_last, avg_monthly_income=income, n_negative_balance=overdrawn,
             n_penalty_interest=penalties, client_age=age, card_type=card,
             avg_salary=d.avg_salary, unemployment_96=d.unemployment_96,
             entrepreneurs_per_1000=d.entrepreneurs_per_1000, urban_ratio=d.urban_ratio,
             crime_rate_per_1000=d.crimes_96 / d.inhabitants * 1000)
    r["payments"] = amount / int(duration)
    r["payment_to_income"] = r["payments"] / max(income, 1)
    r["payment_to_balance"] = r["payments"] / max(balance_mean, 1)
    p = float(model.predict_proba(pd.DataFrame([r])[NUM + CAT])[0, 1])
    band = "High" if p >= HIGH_CUT else "Medium" if p >= LOW_CUT else "Low"
    return p, band, r["payments"]

print("Test:", score("Hl.m. Praha", 100000, 24, 45000, 5000, 40000, 25000, 0, 0, 40, "none"))

### Step 4: Build and open the app

In [ ]:
import gradio as gr

COLOURS = {"Low": "#1E8E3E", "Medium": "#E37400", "High": "#C5221F"}
ACTIONS = {"Low": "Approve", "Medium": "Refer for manual review", "High": "Decline or refer to a credit officer"}
REASONS_TEXT = {"overdrawn": "Account went below zero before the loan",
                "penalty": "Bank charged penalty interest before the loan",
                "low_balance": "Low average balance compared with the monthly payment",
                "neg_min": "Lowest balance is negative"}

def predict(district, amount, duration, balance_mean, balance_min, balance_last, income, overdrawn, penalties, age, card):
    p, band, payment = score(district, amount, duration, balance_mean, balance_min, balance_last, income,
                             overdrawn, penalties, age, card)
    flags = []
    if overdrawn > 0: flags.append(REASONS_TEXT["overdrawn"] + f" ({int(overdrawn)} times)")
    if penalties > 0: flags.append(REASONS_TEXT["penalty"] + f" ({int(penalties)} times)")
    if balance_min < 0: flags.append(REASONS_TEXT["neg_min"])
    if payment / max(balance_mean, 1) > 0.25: flags.append(REASONS_TEXT["low_balance"])
    flag_html = "".join(f"<li>{f}</li>" for f in flags) or "<li>No warning signs in the account history</li>"
    c = COLOURS[band]
    return f'''
    <div style="border-radius:14px;padding:22px;background:{c}14;border:2px solid {c};font-family:sans-serif">
      <div style="font-size:15px;color:#444">Chance this loan will default</div>
      <div style="font-size:54px;font-weight:700;color:{c};line-height:1.1">{p:.0%}</div>
      <div style="height:14px;background:#e6e6e6;border-radius:7px;margin:10px 0 16px">
        <div style="width:{max(p, 0.02):.0%};height:14px;background:{c};border-radius:7px"></div></div>
      <div style="font-size:20px"><b>Risk band:</b> <span style="color:{c};font-weight:700">{band}</span></div>
      <div style="font-size:20px;margin-top:4px"><b>Suggested action:</b> {ACTIONS[band]}</div>
      <div style="font-size:14px;color:#555;margin-top:12px">Monthly payment ≈ {payment:,.0f} CZK</div>
      <div style="font-size:15px;margin-top:12px"><b>What the model noticed:</b><ul style="margin:6px 0 0 18px">{flag_html}</ul></div>
    </div>'''

with gr.Blocks(title="Loan Risk Scorer") as demo:
    gr.Markdown("# 🏦 Loan Risk Scorer\nEnter a loan application and the customer's account history. "
                "The model estimates the chance the loan will **not** be repaid. "
                f"Bands: Low under {LOW_CUT:.0%}, Medium {LOW_CUT:.0%}–{HIGH_CUT:.0%}, High {HIGH_CUT:.0%} and above.")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. The loan")
            district = gr.Dropdown(sorted(DISTRICTS.index), value="Hl.m. Praha", label="Customer's district")
            amount = gr.Number(value=100000, label="Loan amount (CZK)")
            duration = gr.Dropdown([12, 24, 36, 48, 60], value=24, label="Length of loan (months)")
            age = gr.Slider(18, 80, value=40, step=1, label="Customer age")
            card = gr.Dropdown(["none", "junior", "classic", "gold"], value="none", label="Bank card held")
        with gr.Column(scale=1):
            gr.Markdown("### 2. Account history before the loan")
            balance_mean = gr.Number(value=45000, label="Average balance (CZK)")
            balance_min = gr.Number(value=5000, label="Lowest balance (CZK)")
            balance_last = gr.Number(value=40000, label="Latest balance (CZK)")
            income = gr.Number(value=25000, label="Average money coming in per month (CZK)")
            overdrawn = gr.Slider(0, 20, value=0, step=1, label="Times the account went below zero")
            penalties = gr.Slider(0, 10, value=0, step=1, label="Times charged penalty interest")
        with gr.Column(scale=1):
            gr.Markdown("### 3. Result")
            out = gr.HTML()
            btn = gr.Button("Check risk", variant="primary", size="lg")
    inputs = [district, amount, duration, balance_mean, balance_min, balance_last, income, overdrawn, penalties, age, card]
    gr.Examples(label="Try a ready-made customer (click one, then Check risk)",
                examples=[["Hl.m. Praha", 100000, 24, 45000, 5000, 40000, 25000, 0, 0, 40, "none"],
                          ["Hl.m. Praha", 100000, 24, 15000, -500, 8000, 25000, 3, 0, 40, "none"],
                          ["Karvina", 250000, 60, 20000, 1000, 15000, 18000, 0, 2, 30, "none"]],
                example_labels=["Healthy account", "Stretched account", "Big loan, penalties"],
                inputs=inputs)
    btn.click(predict, inputs, out, api_name="predict")
    demo.load(predict, inputs, out)

demo.launch(share=True)   # click the gradio.live link below to open it in a full tab